# 5.3 - Advanced Reasoning: CoT, Self-Consistency, and Knowing When to Stop

**Module 5 - Prompt Engineering** | Netsetos GenAI Engineering

This notebook runs seven reasoning techniques as real Gemini API calls, all built on one shared `ask()` helper: chain-of-thought and structured CoT, self-consistency voting, a native reasoning-model budget, tree-of-thought, a ReAct tool loop, and chain-of-verification. Each is a small, self-contained pattern you can lift straight into your own code, and together they form a decision map - start with the cheapest technique that works and escalate only when it fails.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - a reproducibility note

Before any API calls, a one-line reminder that the notebook's examples are meant to be reproducible. Several later cells pin `temperature=0.0` for exactly this reason. This cell runs no code - it's a comment marking intent.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A no-op marker cell. It documents that the outputs shown throughout are meant to reproduce on rerun (the deterministic cells pin temperature to 0), but the line itself is just a comment and executes nothing.

**How the code works - step by step**
- **The single comment** - notes that the lesson uses seeded/deterministic values so reruns match the shown output.

**In one line:** a comment, not code - it sets the expectation of reproducible runs.

**What you'll see:** (no output - it's a comment, nothing executes)

## Setup - one `ask()` helper we reuse all lesson

Every technique in this notebook is one or more calls to a single wrapper around the Gemini API. We build it once here on the current unified **google-genai** SDK, expose `temperature` and an optional `system` role, and wrap the call in try/except so a single network blip can't crash the multi-call reasoning loops later.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# pip install "google-genai>=1.0.0"
from google import genai
from google.genai import types
import os, re
from collections import Counter

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

def ask(prompt: str, temperature: float = 0.3, system: str = None) -> str:
    """One completion. `system` sets a role; `temperature` controls randomness."""
    cfg = types.GenerateContentConfig(temperature=temperature, system_instruction=system)
    try:
        return client.models.generate_content(
            model="gemini-3-flash", contents=prompt, config=cfg).text.strip()
    except Exception as e:
        return f"[API error: {e}]"

print(ask("Reply with one word: ready"))
# Output: Ready

**Explanation**

The foundation cell: it creates one reusable client and defines `ask()`, a thin one-completion helper that every later cell calls. The two knobs it exposes - `temperature` and `system` - are the exact levers the reasoning patterns turn: temperature drives the sampling that self-consistency and tree-of-thought depend on, and `system` sets a role. This is plumbing, not a technique.

**How the code works - step by step**
- **Imports** - `genai` and `types` from the unified SDK, plus `os`, `re`, and `Counter` (used later to parse answers and tally votes).
- **`genai.Client(api_key=...)`** - one reusable client; the deprecated `google.generativeai` package used a global `configure()` plus a per-call `GenerativeModel`.
- **`ask(prompt, temperature, system)`** - builds a `GenerateContentConfig`, calls `generate_content` on `gemini-3-flash`, and returns the stripped text.
- **`system_instruction`** - routes the optional role into the config.
- **`try/except`** - returns an `"[API error: ...]"` string instead of raising, so a 3-9 call loop survives one bad call.
- **`print(ask(...))`** - a smoke test that the key and client work.

**In one line:** one client, one `ask()`, two knobs - the whole lesson runs on this.

**What you'll see:** Prints `Ready` - the smoke test confirming the client and API key are wired up.

## 1 - Chain-of-thought: make the thinking visible

The simplest reasoning upgrade - force the model to show its work. We send the *same* word problem two ways: 'give only the final number' versus 'think step by step, then end with Answer:', and watch the answer change. It's a controlled A/B - only the reasoning instruction differs, so any change in correctness is caused by the thinking.

In [ ]:
problem = "Apples cost 40 each. Weekends: buy 2 get 1 free. Rahul buys 7 on Saturday. Total paid?"

direct = problem + " Give only the final number."
cot    = problem + " Think step by step, then end with 'Answer: <number>'."

print("direct:", ask(direct, temperature=0.0))
print("cot:   ", ask(cot, temperature=0.0))
# Output: direct: 280                          (7 x 40 - ignores the offer)
# Output: cot:    ...every 3 apples cost 2, so 2 are free, pay for 5,
# Output:         5 x 40 = 200. Answer: 200    (correct)

**Explanation**

A side-by-side A/B on one problem. Both calls use `temperature=0.0` so the contrast is deterministic and repeatable rather than luck. The 'Answer: <number>' instruction plants a stable marker you can later parse - a trick reused in almost every following cell.

**How the code works - step by step**
- **`problem`** - one word problem with a weekend buy-2-get-1-free catch.
- **`direct`** - appends "Give only the final number", leaving no room to reason.
- **`cot`** - appends "Think step by step, then end with 'Answer: <number>'".
- **Both at `temperature=0.0`** - deterministic, so the difference is the reasoning, not randomness.

**In one line:** same problem, only the reasoning instruction differs.

**What you'll see:** `direct: 280` (7x40, ignoring the offer) versus `cot: ...pay for 5, 5 x 40 = 200. Answer: 200` - the step-by-step call reasons through the free apples and gets it right.

## 2 - Structured CoT: split the reasoning from the answer

In production you don't want reasoning mixed into the value you parse. Structured CoT tells the model to reason inside `<thinking>` tags and put only the final result inside `<answer>` tags - then your code extracts just the answer and can still log the reasoning for debugging.

In [ ]:
prompt = """Compute 18% GST on 3 laptops at 45,000 each (intra-state, split CGST/SGST).
Reason inside <thinking> tags. Put ONLY the final breakdown inside <answer> tags."""

resp = ask(prompt, temperature=0.0)
answer = re.search(r"<answer>(.*?)</answer>", resp, re.DOTALL)
print(answer.group(1).strip() if answer else resp)
# Output: Base 135000 | CGST 9% 12150 | SGST 9% 12150 | Total 159300

**Explanation**

The production form of Step 1. The tag order is load-bearing: `<thinking>` before `<answer>` forces the model to work before it commits. A regex then pulls only the `<answer>` block, giving downstream code a clean value while the reasoning stays available in the raw response.

**How the code works - step by step**
- **`prompt`** - a GST calculation with the instruction to reason in `<thinking>` and put ONLY the final breakdown in `<answer>`.
- **`ask(..., temperature=0.0)`** - deterministic call.
- **`re.search(..., re.DOTALL)`** - grabs everything between the `<answer>` tags, across newlines.
- **The `print`** - prints the extracted answer, or falls back to the full response if no tag matched.

**In one line:** reason in one tag, answer in another, parse only the answer.

**What you'll see:** Prints the clean breakdown - `Base 135000 | CGST 9% 12150 | SGST 9% 12150 | Total 159300` - with the reasoning stripped out.

## 3 - Self-consistency: sample, then vote

One chain can slip. Here we sample several chains at a non-zero temperature so each takes a different path, extract every final answer, and return the majority. It trades N-times the cost for reliability on discrete-answer problems.

> **Needs a Gemini API key** - makes N calls (default 5).

In [ ]:
def self_consistency(problem: str, n: int = 5) -> str:
    """Sample n reasoning chains, return the majority final answer."""
    answers = []
    for _ in range(n):
        chain = ask(problem + " Think step by step, end with 'Answer: <value>'.",
                    temperature=0.7)            # >0 so chains differ
        m = re.search(r"Answer:\s*(.+)", chain)
        if m:
            answers.append(m.group(1).strip().rstrip("."))
    return Counter(answers).most_common(1)[0][0]   # the modal answer

problem = "A train goes 60 km/h for 2.5 h, then 80 km/h for 1.5 h. Total distance?"
print(self_consistency(problem, n=5))
# Output: 270 km     (one or two chains may slip to 300; the majority is 270)

**Explanation**

A voting harness around `ask()`. The key knob is `temperature=0.7`: above zero so the N chains actually differ - at temperature 0 you'd get the same chain n times and learn nothing. It reuses the 'Answer:' marker from Step 1, now to collect votes, and tallies them with `Counter`.

**How the code works - step by step**
- **The loop** - runs `n` times, each call sampling one chain at `temperature=0.7`.
- **`re.search(r"Answer:\s*(.+)", chain)`** - extracts and cleans each final answer.
- **`Counter(...).most_common(1)[0][0]`** - returns the modal (majority) answer.
- **Fragility note** - unparseable chains are silently dropped, weakening the vote, and an all-unparseable batch would crash on the empty list; real code counts failures and enforces a minimum quorum.

**In one line:** sample N chains, extract each answer, return the plurality.

**What you'll see:** Prints `270 km` - the correct distance (60x2.5 + 80x1.5), even though one or two individual chains slip to 300; the majority wins.

## 4 - Reasoning models: when the model already thinks

2026's biggest shift - models trained to reason internally, where hand-written 'think step by step' can *hurt*. Here we drop the CoT preamble entirely, state only the goal, and instead set a `thinking_budget` - the dial that controls how much internal reasoning the model spends.

> **Needs a Gemini API key** - uses the native thinking config.

In [ ]:
from google.genai import types

# Let the model think natively. Just state the goal - no "think step by step".
cfg = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=4096))  # tokens of internal thought

resp = client.models.generate_content(
    model="gemini-3-flash",
    contents="Three trucks carry 12.5 t, 9.8 t, 14.2 t. A bridge limit is 35 t. "
             "Which two trucks can cross together with the most load?",
    config=cfg)
print(resp.text)
# Output: 12.5 + 14.2 = 26.7 t and 9.8 + 14.2 = 24.0 t and 12.5 + 9.8 = 22.3 t;
# Output: the heaviest valid pair is 12.5 t + 14.2 t = 26.7 t (under 35 t).

**Explanation**

The contrast to Steps 1-3: instead of prompting the chain, we configure it. `ThinkingConfig(thinking_budget=4096)` tells the model how many tokens of internal thought to spend; the prompt is just the goal. You get a clean answer, and the reasoning happens internally rather than being printed because you asked.

**How the code works - step by step**
- **`GenerateContentConfig(thinking_config=ThinkingConfig(thinking_budget=4096))`** - the reasoning dial, 4096 tokens of internal thought.
- **`contents`** - the bridge/truck problem stated as a plain goal, with no "think step by step".
- **`client.models.generate_content(...)`** - one call; the model reasons natively.
- **`print(resp.text)`** - the clean answer, reasoning done internally.

**In one line:** state the goal, set the budget, let the model reason itself.

**What you'll see:** Prints the pairwise sums and the pick - the heaviest valid pair is 12.5 t + 14.2 t = 26.7 t, under the 35 t bridge limit.

## 5 - Tree-of-thought: explore, evaluate, pick

When there's no single right path, generate several and judge them. This is two calls with deliberately different temperatures: a hot call to produce diverse approaches, then a cold call to evaluate them and pick the best.

> **Needs a Gemini API key** - 2 calls per problem.

In [ ]:
def tree_of_thought(problem: str, k: int = 3) -> str:
    """Generate k approaches (diverse), then judge and pick the best (careful)."""
    ideas = ask(f"Problem: {problem}\nPropose {k} DIFFERENT approaches, each reasoned "
                f"to a conclusion. Label them Approach 1..{k}.",
                temperature=0.7)               # high temp -> diverse ideas
    verdict = ask(f"Problem: {problem}\n\nApproaches:\n{ideas}\n\n"
                  "Evaluate each for correctness and feasibility, then pick the best. "
                  "End with 'Best: <number>' and a one-line reason.",
                  temperature=0.1)             # low temp -> careful judgment
    return verdict

plan = "A startup has a small budget and 3 months to ship an MVP. Strategy?"
print(tree_of_thought(plan))
# Output: Approach 1 (build from scratch)... Approach 2 (no-code)...
# Output: Approach 3 (fork open-source)... Best: 3 - fastest to a testable MVP.

**Explanation**

A generate-then-judge pattern. The two temperatures are the point: `0.7` for the generator (you want breadth - genuinely different approaches, not rewordings) and `0.1` for the judge (you want careful, stable evaluation). Two calls make it pricier than CoT, so it's reserved for open-ended problems with several valid paths.

**How the code works - step by step**
- **Call 1, `temperature=0.7`** - "propose k DIFFERENT approaches, label them 1..k" - deliberately broad.
- **Call 2, `temperature=0.1`** - feeds the approaches back and asks to "evaluate each, pick the best, end with 'Best: <number>'" - careful judgment.
- **The `Best: <number>` marker** - a parseable winner, the same marker trick as Steps 1-3.
- **Returns the verdict text** - approaches plus the pick.

**In one line:** generate hot and broad, judge cold and careful.

**What you'll see:** Prints three labelled approaches (build from scratch / no-code / fork open-source) followed by `Best: 3 - fastest to a testable MVP.`

## 6 - ReAct: reason with tools in a loop

Think -> act -> observe, repeated until the model declares it's done. The model reasons, emits a tool call, your code runs the tool and feeds the result back, and the loop continues under a max-steps guard. This minimal loop is the seed of agents (Module 11).

> **Needs a Gemini API key** - loops up to `max_steps` calls.

In [ ]:
def calc(expr: str) -> str:
    try: return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception: return "calc error"

def react(question: str, max_steps: int = 4) -> str:
    """Think -> Act(calc) -> Observe, looping until 'Final:' or max_steps."""
    history = f"""Answer using the tool calc(expr). Each turn output either
'Action: calc(<expr>)' or 'Final: <answer>'.
Question: {question}"""
    for _ in range(max_steps):                       # max_steps guard = no infinite loop
        out = ask(history, temperature=0.1)
        if "Final:" in out:
            return out.split("Final:")[-1].strip()
        act = re.search(r"calc\((.+?)\)", out)
        obs = calc(act.group(1)) if act else "no tool call"
        history += f"\n{out}\nObservation: {obs}"      # feed result back in
    return "stopped: max steps"

print(react("What is 50000 plus 18% GST on it?"))
# Output: 59000
# (react() returns the text after 'Final:'; internally it ran
#  calc(50000*0.18)->9000.0, then calc(50000+9000)->59000)

**Explanation**

A minimal agent loop. Each turn the model outputs either an `Action: calc(...)` or a `Final:`; your code runs the real tool and appends the result as an 'Observation', so the next turn sees what actually happened. Two guards make it safe-ish: `max_steps` stops runaway loops, and the tool does the exact arithmetic the model is unreliable at.

**How the code works - step by step**
- **`calc(expr)`** - a tiny arithmetic tool that runs in your code, not the model.
- **`react(question, max_steps)`** - builds the protocol prompt, then loops.
- **Inside the loop** - `ask` at `temperature=0.1`; if "Final:" appears, return the answer; else regex the `calc(...)` call, run it, and append the Observation.
- **`max_steps` guard** - returns "stopped: max steps" rather than looping forever - the single most important production rail.
- **Security caveat** - `eval()` on model text is a real code-execution risk and `{"__builtins__":{}}` does NOT make it safe; use an `ast`-based math evaluator in production.

**In one line:** think, call a tool, observe the result, repeat until Final.

**What you'll see:** Prints `59000` - internally it ran calc(50000*0.18)->9000, then calc(50000+9000)->59000, and returned the text after 'Final:'.

## 7 - Chain-of-verification: draft, check, verify, revise

The reliability capstone: draft an answer, turn its claims into fact-check questions, answer those questions *independently* (without showing the draft, so the model can't rubber-stamp itself), then rewrite the draft dropping anything the checks don't support. Four calls, reserved for high-stakes factual output.

> **Needs a Gemini API key** - 4 sequential calls.

In [ ]:
# Chain-of-Verification (CoVe): draft -> plan checks -> verify independently -> revise
draft = ask("List 5 key sections of the Indian IT Act, 2000, with section numbers.")

checks = ask(f"Write 5 yes/no fact-check questions about the claims here:\n{draft}")

# Verify WITHOUT showing the draft - so the model cannot just agree with itself
verified = ask(f"Answer each question independently and factually:\n{checks}")

final = ask(f"Original answer:\n{draft}\n\nVerified facts:\n{verified}\n\n"
            "Rewrite the answer, dropping or correcting any claim the facts do not support.")
print(final)
# Output: a revised list with the unverifiable / wrong section numbers removed

**Explanation**

A four-step self-verification chain, each step one `ask()`. The mechanism is the independence in step 3: because the verifier never sees the original draft, it answers from scratch instead of agreeing with itself. The limit is that it's the same model checking its own knowledge - for statutes or medical/financial numbers you still need an external source of truth (RAG, Module 8).

**How the code works - step by step**
- **`draft`** - the normal, possibly hallucinated answer (IT Act sections).
- **`checks`** - turns the draft's claims into 5 yes/no fact-check questions.
- **`verified`** - answers those questions independently, WITHOUT the draft in context - the whole mechanism.
- **`final`** - reconciles the draft against the verified facts and drops or corrects unsupported claims.

**In one line:** draft -> plan checks -> verify blind -> revise.

**What you'll see:** Prints a revised list with the unverifiable or wrong section numbers removed or corrected.

Seven techniques, one helper: you made the thinking visible (CoT), separated reasoning from the parsed answer (structured CoT), voted across sampled chains (self-consistency), dialed a native reasoning budget instead of hand-writing chains, explored-and-judged (tree-of-thought), looped think-act-observe with a tool (ReAct), and fact-checked a draft against itself (chain-of-verification). The through-line is cost: every pattern past plain CoT adds calls, so match the technique to the task and spend the least that works. Next, Lesson 5.4 turns to context engineering, and the ReAct loop here grows into full agents in Module 11.